In [1]:
import numpy as np
import json
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode
import time

X_pc_full_ea = np.load(r"C:\Users\user\Downloads\GSE148812_clean\v2_pc_input_v2.npy")
with open(r"C:\Users\user\Downloads\GSE148812_clean\v2_pc_col_names_v2.json") as f:
    col_names_ea = json.load(f)

n_nodes = len(col_names_ea)
smoking_idx = col_names_ea.index("smoking_status")
print("Input shape:", X_pc_full_ea.shape)

bk = BackgroundKnowledge()
nodes = [GraphNode(name) for name in col_names_ea]
for i in range(n_nodes - 1):
    bk.add_node_to_tier(nodes[i], 0)
bk.add_node_to_tier(nodes[smoking_idx], 1)

start = time.time()
cg = pc(
    data=X_pc_full_ea,
    alpha=0.001,
    indep_test=fisherz,
    stable=True,
    uc_rule=0,
    uc_priority=2,
    background_knowledge=bk,
    verbose=False,
    show_progress=True,
    node_names=col_names_ea
)
elapsed = time.time() - start
print(f"Completed in {elapsed:.1f}s")
print("Nodes:", len(cg.G.nodes))

c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Input shape: (1595, 53)


Depth=6, working on node 52: 100%|██████████| 53/53 [00:00<00:00, 1084.05it/s]


Completed in 7.9s
Nodes: 53


In [2]:
import networkx as nx

adj = cg.G.graph
n_nodes = len(col_names_ea)
smoking_idx = col_names_ea.index("smoking_status")

edges_directed_ea = []
edges_undirected_ea = []

for i in range(n_nodes):
    for j in range(i+1, n_nodes):
        if adj[i,j] == -1 and adj[j,i] == 1:
            edges_directed_ea.append((col_names_ea[i], col_names_ea[j]))
        elif adj[i,j] == 1 and adj[j,i] == -1:
            edges_directed_ea.append((col_names_ea[j], col_names_ea[i]))
        elif adj[i,j] == -1 and adj[j,i] == -1:
            edges_undirected_ea.append((col_names_ea[i], col_names_ea[j]))

print(f"Directed edges: {len(edges_directed_ea)}")
print(f"Undirected edges: {len(edges_undirected_ea)}")
print(f"Reverse edges: {len([(u,v) for u,v in edges_directed_ea if u=='smoking_status'])}")

print("\nEdges involving smoking_status:")
ea_smoking_snps_v2 = []
for u, v in edges_directed_ea:
    if v == "smoking_status":
        print(f"  {u} → smoking_status")
        ea_smoking_snps_v2.append(u)
for u, v in edges_undirected_ea:
    if "smoking_status" in (u, v):
        other = v if u == "smoking_status" else u
        print(f"  {other} --- smoking_status (undirected)")
        ea_smoking_snps_v2.append(other)

print(f"\nTotal EA SNPs connected to smoking_status: {len(ea_smoking_snps_v2)}")

Directed edges: 53
Undirected edges: 5
Reverse edges: 0

Edges involving smoking_status:
  exm71047-0_B_R_1921357564 → smoking_status
  exm834716-0_B_R_1920965043 → smoking_status
  exm850504-0_B_R_1921041917 → smoking_status
  exm935491-0_T_R_1918372056 → smoking_status
  exm623475-0_B_F_1918444599 → smoking_status

Total EA SNPs connected to smoking_status: 5
